<a href="https://colab.research.google.com/github/normieparv00/NeuroNexus/blob/main/TITANIC_SURVIVAL_PREDICTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os

In [31]:
df = pd.read_csv('/tested.csv')

In [32]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [52]:
#now we need to fill the missing information that might be present in the given dataset

df.fillna({"Age": df["Age"].median(),
           "Fare": df["Fare"].median(),
           "Embarked": df["Embarked"].mode()[0]}, inplace=True)

#Here we are taking the median of the age and fare, but it depends on case to case basis on the data and requirements.

In [53]:
#now we need to encode the categorical features as the computer does not understand strings, we need to convert it into numbers

le = LabelEncoder()
df["Sex"] = le.fit_transform(df["Sex"])
df["Embarked"] = le.fit_transform(df["Embarked"])

In [44]:
text_cols = df.select_dtypes(include=["object"]).columns
df.drop(columns=text_cols, inplace=True)

In [45]:
#Doing our train-test split

In [50]:
X = df.drop(["Survived", "PassengerId"], axis=1)
y = df["Survived"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [51]:
#Training a Randon FOrest model

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluating our model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        34

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84



Here the accuracy, precision, recall and f1-score is perfect 1.0
This raises suspisions that there might be data leaks or label leakage via encoded columns.

To test wether or not this is the case, we would need to do some tests below

In [59]:
#Making sure that training and testing sets are different

print(X_train.head())
print(X_test.head())

     Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
336       2    1  32.0      0      0  13.0000         2
31        2    1  24.0      2      0  31.5000         2
84        2    1  27.0      0      0  10.7083         1
287       1    1  24.0      1      0  82.2667         2
317       2    1  19.0      0      0  10.5000         2
     Pclass  Sex   Age  SibSp  Parch      Fare  Embarked
321       3    1  25.0      0      0    7.2292         0
324       1    0  39.0      0      0  211.3375         2
388       3    1  21.0      0      0    7.7500         1
56        3    1  35.0      0      0    7.8958         2
153       3    0  36.0      0      2   12.1833         2


In [60]:
#Making sure that X does not have any target related columns which in our case is the "Survived column"

print(X.columns)

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')


In [61]:
#We can also check the dimensions to make sure that our model is training and testing on different sets

print(X_train.shape)
print(X_test.shape)

(334, 7)
(84, 7)


In [63]:
print(y_train.shape)
print(y_test.shape)

(334,)
(84,)


The model achieved perfect performance metrics (accuracy, precision, recall, F1-score), and after thorough verification, no data leakage or overlap between training and validation sets was found. All features were properly encoded, and the target variable was excluded from the input. A Random Forest classifier was used due to its robustness, ability to handle both numerical and categorical data, and strong performance on tabular datasets like Titanic.